# 실습 3 — 모델 서빙 (vLLM)

파인튜닝한 모델을 **vLLM**으로 서빙하여, 실제 서비스처럼 API를 통해 모델에 요청을 보내고 응답을 받습니다.

| 항목 | 내용 |
|---|---|
| **서빙 엔진** | vLLM |
| **사용 모델** | 실습 2에서 파인튜닝한 Qwen2.5-3B |
| **실습 내용** | Python API 추론, OpenAI 호환 서버, 스트리밍 |

> **왜 서빙 엔진이 필요한가?**  
> `transformers pipeline`은 요청을 하나씩 순서대로 처리합니다.  
> vLLM은 **여러 요청을 동시에 처리**하고 GPU 메모리를 효율적으로 관리하여 2~5배 빠릅니다.

## ⚙️ Colab GPU 설정

실습 3도 GPU가 필요합니다.

**설정 방법:**
1. 상단 메뉴 → **런타임** 클릭
2. **런타임 유형 변경** 클릭
3. 하드웨어 가속기 → **T4 GPU** 선택
4. **저장** 후 런타임 재시작

> ⚠️ 실습 2와 다른 Colab 세션이라면 모델을 다시 로딩하거나 HuggingFace Hub에서 불러와야 합니다.

In [ ]:
import torch
print(f"GPU 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 📦 라이브러리 설치

In [ ]:
# 1. 기존에 설치된 충돌 패키지 완전 삭제
!pip uninstall -y vllm torch torchvision torchaudio

# 2. CUDA 12.4 버전에 맞는 PyTorch 강제 설치
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# 3. vLLM 및 기타 실습 필수 라이브러리 재설치
!pip install -q vllm pyngrok openai peft accelerate bitsandbytes

In [ ]:
pip uninstall -y torchao

## 1단계. 파인튜닝된 모델 준비

vLLM은 LoRA 어댑터와 베이스 모델이 분리된 상태보다  
**병합(merge)된 단일 모델 파일**을 더 간편하게 사용할 수 있습니다.

```
BeforeL 베이스 모델 + LoRA 어댑터 (분리 상태)
After:  베이스 모델에 LoRA 가중치가 합쳐진 단일 모델
```

---
### ✏️ 빈칸 1, 2 — 어댑터 병합 및 저장

---

### 💡  HuggingFace Hub에서 불러오기

실습 2에서 허브에 업로드한 경우, 새 세션에서 아래 코드로 불러올 수 있습니다.


In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, os

HF_ADAPTER_ID = "zzmnxn/yoda-lora-qwen"  # 어댑터 repo
BASE_MODEL_ID  = "unsloth/Qwen2.5-1.5B-Instruct"  # 베이스 모델
MERGED_PATH    = "./merged-model"

# 1. 베이스 모델 로드
print("베이스 모델 로딩 중...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

# 2. LoRA 어댑터 로드 후 병합
print("LoRA 어댑터 병합 중...")
model = PeftModel.from_pretrained(base_model, HF_ADAPTER_ID)
merged_model = model.merge_and_unload()

# 3. 병합된 모델 저장
print("저장 중...")
merged_model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)

print(f"✅ 병합 완료: {MERGED_PATH}")
for f in sorted(os.listdir(MERGED_PATH)):
    size = os.path.getsize(f"{MERGED_PATH}/{f}")
    print(f"  {f}: {size/1024/1024:.1f} MB")



## 2단계. vLLM Python API (방법 A)

서버를 따로 띄우지 않고 Python 코드 안에서 직접 vLLM을 사용합니다.  
Colab 실습에 가장 적합한 방법입니다.

---
### ✏️ 빈칸 3, 4 — LLM 초기화 및 SamplingParams 설정

---

In [ ]:
import torch, gc

gc.collect()
torch.cuda.empty_cache()
print(f"남은 메모리: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

from vllm import LLM, SamplingParams

# ✏️ 빈칸 3: 병합된 모델 경로를 넣어 LLM을 초기화하세요
# 힌트: LLM(model="모델경로")
llm = LLM(
    model=MERGED_PATH,
    dtype="float16",
    gpu_memory_utilization=0.75,
    max_model_len=2048,
)

# ✏️ 빈칸 4: 추론 설정을 채워보세요
# - temperature: 0이면 항상 동일한 결과 (재현성), 높을수록 창의적
# - max_tokens: 최대 생성 토큰 수
sampling_params = SamplingParams(
    temperature=0,  # ✏️ 빈칸 4a: 0 ~ 1 사이 값 (힌트: 일관된 QA 결과를 원하면?)
    max_tokens=60,   # ✏️ 빈칸 4b: QA 답변은 보통 짧으므로 적절한 값은?
)

print("✅ vLLM 초기화 완료")

### 📊 temperature 값의 의미

| temperature | 특징 | 적합한 상황 |
|---|---|---|
| `0.0` | 항상 가장 확률 높은 토큰 선택, 결과 동일 | 정확한 QA, 분류 |
| `0.3~0.7` | 약간의 다양성, 자연스러운 응답 | 일반 대화 |
| `1.0+` | 창의적이지만 엉뚱할 수 있음 | 창작, 브레인스토밍 |

### ✏️ 빈칸 5 — 추론 요청 구성

여러 질문을 한 번에 보내는 배치 추론을 해봅니다.

---

In [ ]:
# Alpaca 프롬프트 템플릿 (실습 2와 동일)
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
"""

# 테스트 케이스
test_cases = [
    {"sentence": "The Force is strong with you.",
     "yoda": "Strong with you, the Force is."},
    {"sentence": "You must learn patience.",
     "yoda": "Patience, you must learn."},
    {"sentence": "Knowledge is the path to wisdom.",
     "yoda": "The path to wisdom, knowledge is."},
    {"sentence": "Fear leads to anger.",
     "yoda": "To anger, fear leads."},
    {"sentence": "Do or do not, there is no try.",
     "yoda": "Do or do not, there is no try."},
]

# 프롬프트 리스트 생성
prompts = [
    alpaca_prompt.format(
        "Translate the following English sentence into Yoda's speaking style.",
        tc["sentence"],
        ""
    )
    for tc in test_cases
]

print(f"✅ 프롬프트 {len(prompts)}개 생성 완료")

In [ ]:
# 배치 추론 실행
outputs = llm.generate(prompts, sampling_params)

print("=" * 60)
print("vLLM Python API 추론 결과")
print("=" * 60)
for i, output in enumerate(outputs):
    answer = output.outputs[0].text.strip()
    print(f"질문: {test_cases[i]['sentence']}")
    print(f"답변: {answer}")
    print("-" * 60)

### 📊 결과 해석

- vLLM은 여러 프롬프트를 **동시에 처리**합니다 (배치 추론).
- `transformers`로 하나씩 처리하는 것보다 훨씬 빠릅니다.
- `output.outputs[0].text`로 생성된 텍스트에 접근합니다.